# AgroScan AI — Entrenamiento del modelo de hojas de frijol

Entrena un clasificador de **3 clases** (`sana`, `mancha_angular`, `roya`) con el dataset público
**iBean** (Makerere AI Lab) y lo exporta a TensorFlow Lite para la app.

**Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)*.
Después: *Entorno de ejecución → Ejecutar todo*. Tarda unos 10–20 minutos.

Al final se descargan 3 archivos:
- `modelo_frijol.tflite` → va a `app/src/main/assets/`
- `etiquetas_frijol.txt` → va a `app/src/main/assets/`
- `metricas_modelo.json` → métricas para el artículo (no va en la app)

## 1. Configuración

In [ ]:
!pip install -q datasets

import json
import numpy as np
import tensorflow as tf
from datasets import load_dataset
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "NO (activa la GPU en el menú)")

SEMILLA = 42                      # fija para que el entrenamiento sea reproducible
tf.keras.utils.set_random_seed(SEMILLA)

TAM = 224                         # tamaño de entrada del modelo (224x224 RGB)
LOTE = 32
ETIQUETAS = ["sana", "mancha_angular", "roya"]   # orden de la salida del modelo (= CatalogoFrijol)

## 2. Dataset iBean

Se descarga automáticamente de Hugging Face (`AI-Lab-Makerere/beans`). Ya viene dividido en
entrenamiento, validación y prueba.
Cita: *Makerere AI Lab (2020). iBean: Bean disease dataset.* https://github.com/AI-Lab-Makerere/ibean

In [ ]:
beans = load_dataset("AI-Lab-Makerere/beans")

nombres_originales = beans["train"].features["labels"].names
num_fotos = {s: len(beans[s]) for s in ["train", "validation", "test"]}
print("Clases originales:", nombres_originales)
print(num_fotos)

# El dataset usa el orden [angular_leaf_spot, bean_rust, healthy];
# lo reordenamos a [sana, mancha_angular, roya].
assert nombres_originales == ["angular_leaf_spot", "bean_rust", "healthy"]
REMAP = tf.constant([1, 2, 0])

def a_dataset(split):
    imagenes, etiquetas = [], []
    for ejemplo in beans[split]:
        foto = ejemplo["image"].convert("RGB").resize((TAM, TAM), Image.BILINEAR)
        imagenes.append(np.asarray(foto, dtype=np.uint8))
        etiquetas.append(ejemplo["labels"])
    return tf.data.Dataset.from_tensor_slices((np.stack(imagenes), np.array(etiquetas)))

entren, valid, prueba = (a_dataset(s) for s in ["train", "validation", "test"])

## 3. Preparar las imágenes

El modelo recibe píxeles RGB **0–255** sin normalizar (la normalización va dentro del modelo).
Al entrenamiento se le aplican giros y cambios de brillo para que no dependa de cómo se tomó la foto.

In [ ]:
def preparar(img, etiqueta):
    return tf.cast(img, tf.float32), tf.gather(REMAP, etiqueta)

def aumentar(img, etiqueta):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.rot90(img, tf.random.uniform([], 0, 4, dtype=tf.int32))
    img = tf.image.random_brightness(img, 25.0)
    img = tf.image.random_contrast(img, 0.85, 1.15)
    return tf.clip_by_value(img, 0.0, 255.0), etiqueta

AUTO = tf.data.AUTOTUNE
ds_entren = (entren.map(preparar, AUTO).cache().shuffle(1000, seed=SEMILLA)
             .map(aumentar, AUTO).batch(LOTE).prefetch(AUTO))
ds_valid  = valid.map(preparar, AUTO).cache().batch(LOTE).prefetch(AUTO)
ds_prueba = prueba.map(preparar, AUTO).cache().batch(LOTE)

## 4. Modelo (transfer learning con MobileNetV2)

Se usa MobileNetV2 preentrenado en ImageNet y solo se entrena la última capa.

In [ ]:
base = tf.keras.applications.MobileNetV2(
    input_shape=(TAM, TAM, 3), include_top=False, weights="imagenet")
base.trainable = False

entrada = tf.keras.Input((TAM, TAM, 3))                          # RGB 0–255
x = tf.keras.layers.Rescaling(1 / 127.5, offset=-1)(entrada)     # a [-1, 1], como espera MobileNetV2
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
salida = tf.keras.layers.Dense(len(ETIQUETAS), activation="softmax")(x)
modelo = tf.keras.Model(entrada, salida)

parar = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

modelo.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
               loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist1 = modelo.fit(ds_entren, validation_data=ds_valid, epochs=15, callbacks=[parar])

## 5. Ajuste fino

Se descongelan las últimas capas de MobileNetV2 y se entrenan con una tasa de aprendizaje muy baja.

In [ ]:
base.trainable = True
for capa in base.layers[:100]:
    capa.trainable = False

modelo.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
               loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist2 = modelo.fit(ds_entren, validation_data=ds_valid, epochs=15, callbacks=[parar])

## 6. Evaluación con el conjunto de prueba

Fotos que el modelo **nunca vio** durante el entrenamiento. Estas son las cifras para el artículo.

In [ ]:
y_real = np.concatenate([y.numpy() for _, y in ds_prueba])
y_pred = modelo.predict(ds_prueba, verbose=0).argmax(axis=1)

exactitud = float((y_real == y_pred).mean())
reporte = classification_report(y_real, y_pred, target_names=ETIQUETAS, digits=4, output_dict=True)
matriz = confusion_matrix(y_real, y_pred)

print(f"Exactitud en prueba: {exactitud:.4f} ({len(y_real)} fotos)\n")
print(classification_report(y_real, y_pred, target_names=ETIQUETAS, digits=4))
print("Matriz de confusión (filas = real, columnas = predicho)", ETIQUETAS)
print(matriz)

## 7. Exportar a TensorFlow Lite y comprobarlo

In [ ]:
modelo.export("modelo_guardado")
convertidor = tf.lite.TFLiteConverter.from_saved_model("modelo_guardado")
with open("modelo_frijol.tflite", "wb") as f:
    f.write(convertidor.convert())
with open("etiquetas_frijol.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(ETIQUETAS) + "\n")

# Comprobar que el .tflite da los mismos resultados que el modelo original
interprete = tf.lite.Interpreter(model_path="modelo_frijol.tflite")
interprete.allocate_tensors()
ent = interprete.get_input_details()[0]
sal = interprete.get_output_details()[0]
print("Entrada:", ent["shape"], ent["dtype"].__name__, "| Salida:", sal["shape"], sal["dtype"].__name__)

aciertos = 0
for img, y in prueba.map(preparar):
    interprete.set_tensor(ent["index"], img.numpy()[None])
    interprete.invoke()
    aciertos += int(interprete.get_tensor(sal["index"]).argmax() == int(y))
exactitud_tflite = aciertos / len(y_real)
print(f"Exactitud del .tflite en prueba: {exactitud_tflite:.4f}")

## 8. Guardar métricas y descargar los archivos

In [ ]:
metricas = {
    "dataset": "iBean (Makerere AI Lab), Hugging Face AI-Lab-Makerere/beans",
    "fotos": num_fotos,
    "arquitectura": "MobileNetV2 (ImageNet) + Dense softmax, transfer learning + ajuste fino",
    "entrada": {"tamano": [TAM, TAM, 3], "tipo": "float32", "pixeles": "RGB 0-255 (normalización dentro del modelo)"},
    "salida_orden": ETIQUETAS,
    "epocas": {"base_congelada": len(hist1.history["loss"]), "ajuste_fino": len(hist2.history["loss"])},
    "exactitud_prueba": exactitud,
    "exactitud_prueba_tflite": exactitud_tflite,
    "reporte_por_clase": reporte,
    "matriz_confusion": {"filas_real_columnas_predicho": ETIQUETAS, "valores": matriz.tolist()},
    "tensorflow": tf.__version__,
    "semilla": SEMILLA,
}
with open("metricas_modelo.json", "w", encoding="utf-8") as f:
    json.dump(metricas, f, indent=2, ensure_ascii=False)

from google.colab import files
for archivo in ["modelo_frijol.tflite", "etiquetas_frijol.txt", "metricas_modelo.json"]:
    files.download(archivo)

## Importante para el artículo

Las fotos de iBean se tomaron en campos de **Uganda**. Que el modelo acierte con esas fotos
no garantiza que acierte con las de tu parcela (otra variedad, luz y cámara). Para que el
resultado sea verificable, conviene tomar fotos propias de la parcela, etiquetadas por alguien
que sepa identificar las enfermedades, y medir la exactitud del modelo también con ellas.